# Ransac and Outlier Removal

In [26]:
from typing import Callable, Optional
import numpy as np
from numpy.random import Generator
from pydrake.all import (
    Cylinder,
    PointCloud,
    Rgba,
    RigidTransform,
    RollPitchYaw,
    RotationMatrix,
    StartMeshcat,
)

from manipulation import FindResource
from manipulation.exercises.grader import Grader
from manipulation.exercises.pose.test_ransac import TestRANSAC

In [27]:
# Start the visualizer.
meshcat = StartMeshcat()
meshcat.SetProperty("/Background", "visible", False)
meshcat.SetProperty("/Cameras/default/rotated/<object>", "zoom", 10.5)

INFO:drake:Meshcat listening for connections at http://localhost:7005


In [38]:
# Visualize Stanford Bunny
xyzs = np.load(FindResource("models/bunny/bunny.npy"))

# point clouds of planar surface
grid_spec = 50
xy_axis = np.linspace(-0.5, 0.5, grid_spec)
plane_x, plane_y = np.meshgrid(xy_axis, xy_axis)
points_plane_xy = np.c_[plane_x.flatten(), plane_y.flatten(), np.zeros(grid_spec**2)]
bunny_w_plane = np.c_[points_plane_xy.T, xyzs]


def fit_plane(xyzs: np.ndarray) -> np.ndarray:
    """
    Fits a plane to a set of 3D points using Singular Value Decomposition (SVD).

    Args:
        xyzs (numpy.ndarray): A (3, N) numpy array where N is the number of points.

    Returns:
        numpy.ndarray: A (4,) numpy array representing the plane equation coefficients [a, b, c, d]
                       such that ax + by + cz + d = 0.
    """
    # Ensure the input is a numpy array
    xyzs = np.asarray(xyzs)

    # Check if the input has the correct shape
    if xyzs.shape[0] != 3:
        raise ValueError("Input array must have shape (3, N)")

    # Compute the centroid of the point cloud
    center = np.mean(xyzs, axis=1)

    # Center the point cloud at the origin
    centered_xyzs = xyzs.T - center

    # Perform Singular Value Decomposition
    U, S, Vt = np.linalg.svd(centered_xyzs)

    # The normal to the plane is the last row of Vt (or the last column of V)
    normal = Vt[-1]

    # Compute the plane equation coefficient d
    d = -center.dot(normal)

    # Combine the normal vector and d to form the plane equation
    plane_equation = np.hstack([normal, d])

    return plane_equation


def visualize_plane(
    abcd: np.ndarray, name: str, center: Optional[np.ndarray] = None
) -> None:
    """
    Visualizes a plane and its normal vector in the MeshCat environment.

    Args:
        abcd (numpy.ndarray): A (4,) numpy array representing the plane equation coefficients [a, b, c, d]
                              such that ax + by + cz + d = 0.
        name (str): The name of the visualization object.
        center (numpy.ndarray, optional): The center of the plane. Defaults to None.
    """
    # The normal to the plane is the first three elements of the plane equation coefficients
    normal = np.array(abcd[:3]).astype(float)
    # Normalize the normal vector
    normal /= np.linalg.norm(normal)
    # The distance from the origin to the plane is the last element of the plane equation coefficients
    d = -abcd[3] / np.linalg.norm(normal)

    # Create a rotation matrix R with the normal as the third column
    z = normal
    R = np.eye(3)
    R[:, 2] = z
    # If the x component of the normal is close to zero, choose a different vector for x
    if abs(z[0]) < 1e-8:
        x = np.array([0, -z[2], z[1]])
    else:
        x = np.array([-z[1], z[0], 0])
    # Normalize the x vector
    x /= np.linalg.norm(x)
    # The y vector is the cross product of z and x
    R[:, 0] = x
    R[:, 1] = np.cross(z, x)
    X = RigidTransform(RotationMatrix(R), d * normal)

    # Set the visualization objects in the MeshCat environment
    meshcat.SetObject(name + "/plane", Cylinder(0.1, 0.005))
    meshcat.SetTransform(name + "/plane", X)

    meshcat.SetObject(name + "/normal", Cylinder(0.001, 0.2), Rgba(0, 0, 1))
    meshcat.SetTransform(name + "/normal", X @ RigidTransform([0, 0, 0.1])) # type: ignore


def visualize_point_clouds(points, name):
    cloud = PointCloud(points.shape[1])
    cloud.mutable_xyzs()[:] = points
    meshcat.SetObject(name, cloud, point_size=0.01, rgba=Rgba(1.0, 0, 0))

# Problem Description
In the lecture, we learned about the RANSAC algorithm. In this exercise, you will implement the RANSAC algorithm to separate the Stanford bunny from its environment!

**These are the main steps of the exercise:**
1. Implement the `ransac` method.
2. Implement the `remove_plane` method to remove the points that belong to the planar surface.

Let's first visualize the point clouds of Stanford bunny in meshcat!

In [39]:
visualize_point_clouds(bunny_w_plane, "bunny_w_plane")

You should notice that now there is a planar surface underneath the bunny. You may assume the bunny is currently placed on a table, where the planar surface is the tabletop. In this exercise, your objective is to remove the planar surface.

A straightforward way to achieve a better fit is to remove the planar surface underneath the bunny. To do so, we provide you a function to fit a planar surface.

Recall that a plane equation is of the form
$$a x + b y + c z + d = 0$$
where $[a,b,c]^T$ is a vector normal to the plane and (if it's a unit normal) $d$ is the negative of the distance from the origin to the plane in the direction of the normal.  We'll represent a plane by the vector $[a,b,c,d]$.

The fitted planes are shown as translucent disks of radius $r$ centered at the points. The gray lines represent the planes' normals.

In [40]:
plane_equation = fit_plane(bunny_w_plane)
print(plane_equation)
visualize_plane(plane_equation, "naive_plane")

[-0.00617359 -0.03380922 -0.99940924  0.03336162]


You should notice that the planar surface cannot be fitted exactly either. This is because it takes account of all points in the scene to fit the plane. Since a significant portion of the point cloud belongs to the bunny, the fitted plane is noticeably elevated above the ground.

To improve the result of the fitted plane, you will use RANSAC!

## RANSAC
With the presence of outliers (bunny), we can use RANSAC to get more reliable estimates. The idea is to fit a plane using many random choices of a minimal set of points (3), fit a plane for each one, count how many points are within a distance tolerance to that plane (the inliers), and return the estimate with the most inliers.

**Complete the function `ransac`.  It takes a data matrix, a tolerance, a value of iterations, and a model regressor. It returns an equation constructed by the model regressor and a count of inliers.**

In [ ]:
def ransac(
    point_cloud: np.ndarray,
    model_fit_func: Callable,
    rng: Generator,
    tolerance: float = 1e-3,
    max_iterations: int = 500,
) -> tuple[int, np.ndarray | None]:
    """
    Args:
      point_cloud is (3, N) numpy array
      tolerance is a float
      rng is a random number generator
      max_iterations is a (small) integer
      model_fit_func: the function to fit the model (point clouds)

    Returns:
      (4,) numpy array
    """
    best_ic = 0  # inlier count
    best_model = np.ones(4)  # plane equation ((4,) array)
    N = point_cloud.shape[1]
    min_inliers = max(3, int(0.1 * N))  # 最小内点阈值，例如 10% of N
    ##################
    # your code here
    for _ in range(max_iterations):
        # 正确采样：选 3 个独特索引
        indices = rng.choice(N, 3, replace=False)
        sample_points = point_cloud[:, indices]  # (3,3)

        # 拟合平面
        plane_equation = model_fit_func(sample_points)
        if plane_equation is None: continue  # 如果拟合失败

        # 计算距离（绝对值 + 归一化）
        normal = plane_equation[:3]
        norm = np.linalg.norm(normal)
        if norm < 1e-6: continue  # 无效法向量
        distances = np.abs((normal @ point_cloud + plane_equation[3]) / norm)  # (N,)
        # print(distances)
        # 统计内点数
        inliers = np.sum(distances < tolerance)

        # 更新最佳（如果内点更多）
        if inliers > best_ic:
            best_ic = inliers
            best_model = plane_equation
    
    # 如果内点不足，返回 None
    if best_ic < min_inliers:
        return 0, best_model
    ##################

    return best_ic, best_model

Now you should have a lot better estimate of the planar surface with the use of RANSAC! Let's visualize the plane now!

In [ ]:
rng = np.random.default_rng(135)  # random number generator
inlier_count, ransac_plane = ransac(bunny_w_plane, fit_plane, rng, 0.001, 500)
# print(ransac_plane)
visualize_plane(ransac_plane, "ransac_plane", center=[0, 0, -ransac_plane[-1]]) # type: ignore

[0.46554751 0.45280605 0.44006458 ... 0.036973   0.0090319  0.01252026]
[0.61965374 0.60412342 0.58859311 ... 0.07212597 0.04110705 0.04124643]
[0.62999291 0.61661228 0.60323166 ... 0.04544366 0.02419391 0.01225118]
[0.50016233 0.48860853 0.47705473 ... 0.02273946 0.03947593 0.05230231]
[0.16877925 0.17767189 0.18656453 ... 0.00974156 0.00772189 0.02937766]
[0.65121918 0.64259696 0.63397473 ... 0.0478931  0.02635972 0.00918661]
[0.27351242 0.27978196 0.28605149 ... 0.0072061  0.00709685 0.03200623]
[0.05790531 0.07079169 0.08367807 ... 0.01638089 0.02610047 0.00436353]
[0.14095877 0.14377786 0.14659695 ... 0.0249652  0.0180777  0.03737039]
[0.1557717  0.15585061 0.15592952 ... 0.01332747 0.00836578 0.023974  ]
[0.62389181 0.60620925 0.58852669 ... 0.03413778 0.0073659  0.00314968]
[0.56810508 0.5519482  0.53579132 ... 0.03166729 0.00933996 0.00398302]
[4.87953139e-01 4.69545029e-01 4.51136920e-01 ... 3.09122175e-02
 2.67197079e-04 8.74847371e-03]
[0.65267311 0.63487357 0.61707403 ... 0

## Remove Planar Surface

Now all you need to do is to remove the points that belong to the planar surface. You may do so by rejecting all points that are
$$|| a x + b y + c z + d || < tol$$

Note that since you are fitting a plane, the bunny is this case is the "outlier". Your job, however, is to keep the bunny and remove the planar surface.

**Complete the function below to remove the points that belongs to the planar surface**.

In [61]:
def remove_plane(
    point_cloud: np.ndarray, ransac: Callable, rng: Generator, tol: float = 1e-4
) -> np.ndarray:
    """
    Args:
        point_cloud: 3xN numpy array of points
        ransac: The RANSAC function to use (call ransac(args))
        rng is a random number generator
        tol: points less than this distance tolerance should be removed
    Returns:
        point_cloud_wo_plane: 3xN numpy array of points
    """
    point_cloud_wo_plane = np.zeros((3, 100))

    N = point_cloud.shape[1]
    min_inliers = max(3, int(0.1 * N))  # 最小内点阈值，确保检测到可靠平面
    best_ic, best_model = ransac(point_cloud, fit_plane, rng, 0.001, 500)
    
    # 检查：如果没有可靠平面（e.g., 纯 Bunny 无平面），返回原点云
    if best_model is None or best_ic < min_inliers:
        return point_cloud
    
    normal = best_model[:3]
    norm = np.linalg.norm(normal)
    if norm < 1e-6:
        return point_cloud  # 无效模型
    distances = np.abs((normal @ point_cloud + best_model[3]) / norm)  # (N,)
    # 保留 outlier（距离 >= tol，如 Bunny 点），移除 inlier（距离 < tol，如平面点）
    mask = distances >= tol

    point_cloud_wo_plane = point_cloud[:, mask]  # (3, M)，M 是保留点数

    return point_cloud_wo_plane

In [62]:
x = np.array([1,2,3,4]) # type: ignore
y = np.array([2,5,2,4]) # type: ignore
ds = np.stack([x, y])
print(ds)
mask = [True, False,True,True]
print(ds[:,mask])

[[1 2 3 4]
 [2 5 2 4]]
[[1 3 4]
 [2 2 4]]


In [63]:
meshcat.Delete()
rng = np.random.default_rng(135)  # random number generator
bunny_wo_plane = remove_plane(bunny_w_plane, ransac, rng)
visualize_point_clouds(bunny_wo_plane, "bunny")

[0.46554751 0.45280605 0.44006458 ... 0.036973   0.0090319  0.01252026]
[0.61965374 0.60412342 0.58859311 ... 0.07212597 0.04110705 0.04124643]
[0.62999291 0.61661228 0.60323166 ... 0.04544366 0.02419391 0.01225118]
[0.50016233 0.48860853 0.47705473 ... 0.02273946 0.03947593 0.05230231]
[0.16877925 0.17767189 0.18656453 ... 0.00974156 0.00772189 0.02937766]
[0.65121918 0.64259696 0.63397473 ... 0.0478931  0.02635972 0.00918661]
[0.27351242 0.27978196 0.28605149 ... 0.0072061  0.00709685 0.03200623]
[0.05790531 0.07079169 0.08367807 ... 0.01638089 0.02610047 0.00436353]
[0.14095877 0.14377786 0.14659695 ... 0.0249652  0.0180777  0.03737039]
[0.1557717  0.15585061 0.15592952 ... 0.01332747 0.00836578 0.023974  ]
[0.62389181 0.60620925 0.58852669 ... 0.03413778 0.0073659  0.00314968]
[0.56810508 0.5519482  0.53579132 ... 0.03166729 0.00933996 0.00398302]
[4.87953139e-01 4.69545029e-01 4.51136920e-01 ... 3.09122175e-02
 2.67197079e-04 8.74847371e-03]
[0.65267311 0.63487357 0.61707403 ... 0

In [64]:
# Use this to test your implementation if you want!
Grader.grade_output([TestRANSAC], [locals()], "results.json")
Grader.print_test_results("results.json")

Total score is 6/6.

Score for Test outlier removal is 2/2.
- [0.46554751 0.45280605 0.44006458 ... 0.036973   0.0090319  0.01252026]
[0.61965374 0.60412342 0.58859311 ... 0.07212597 0.04110705 0.04124643]
[0.62999291 0.61661228 0.60323166 ... 0.04544366 0.02419391 0.01225118]
[0.50016233 0.48860853 0.47705473 ... 0.02273946 0.03947593 0.05230231]
[0.16877925 0.17767189 0.18656453 ... 0.00974156 0.00772189 0.02937766]
[0.65121918 0.64259696 0.63397473 ... 0.0478931  0.02635972 0.00918661]
[0.27351242 0.27978196 0.28605149 ... 0.0072061  0.00709685 0.03200623]
[0.05790531 0.07079169 0.08367807 ... 0.01638089 0.02610047 0.00436353]
[0.14095877 0.14377786 0.14659695 ... 0.0249652  0.0180777  0.03737039]
[0.1557717  0.15585061 0.15592952 ... 0.01332747 0.00836578 0.023974  ]
[0.62389181 0.60620925 0.58852669 ... 0.03413778 0.0073659  0.00314968]
[0.56810508 0.5519482  0.53579132 ... 0.03166729 0.00933996 0.00398302]
[4.87953139e-01 4.69545029e-01 4.51136920e-01 ... 3.09122175e-02
 2.671970

---

# VERIFICATION IN GRADESCOPE 

**Prerequisites:** You must complete ALL the TODOs above before these verification exercises will work!

**Instructions:** Implement the exercises below. Copy the exact numerical values (to 4 decimal places) for your verification keys, which you can copy/paste to Gradescope.

## Verification 1: RANSAC function

**Task:** Compute the RANSAC algorithm result for the same bunny point cloud as defined above (`bunny_w_plane`), with
- `rng = np.random.default_rng(120)` as random number generator
- tolerance = 1e-2
- max num of iterations = 200

**Question:** For the condition above, run the RANSAC algorithm, what is the count of inliers and the parameter of the equation constructed by the model regressor?

In [65]:
rng = np.random.default_rng(120)  # random number generator
ransac(bunny_w_plane, fit_plane, rng, 1e-2, 200)

[0.62340903 0.61632702 0.60924501 ... 0.07225798 0.05039054 0.03572844]
[0.16422836 0.1654593  0.16669025 ... 0.0341678  0.02934603 0.04787035]
[0.68791923 0.66957684 0.65123445 ... 0.07487202 0.04336961 0.04183034]
[0.22856055 0.21092869 0.19329682 ... 0.0474456  0.02690593 0.04592335]
[0.65349544 0.64163662 0.62977779 ... 0.12388994 0.09592295 0.09169807]
[0.42698791 0.4082264  0.38946489 ... 0.0405933  0.01804866 0.02547785]
[0.61093318 0.59201103 0.57308888 ... 0.02276711 0.0092372  0.00816207]
[0.12210207 0.11690623 0.1117104  ... 0.03788035 0.03734212 0.04424731]
[0.20506495 0.18740018 0.16973541 ... 0.02194096 0.00135764 0.02029956]
[0.54190273 0.52166911 0.50143549 ... 0.08137851 0.05113127 0.06077214]
[0.10612572 0.10363368 0.10114165 ... 0.0864406  0.0745236  0.08467961]
[0.1937652  0.19997828 0.20619135 ... 0.06062201 0.05691419 0.0452136 ]
[0.21048735 0.20453016 0.19857297 ... 0.05030863 0.05310275 0.06367946]
[0.47917254 0.46013404 0.44109554 ... 0.04054816 0.01643126 0.02

(np.int64(3640), array([-0.0059668 ,  0.00954688,  0.99993663, -0.00200923]))

## Verification 2: Remove Planar Surface function

**Task:** Now still with the same bunny point cloud as defined above (`bunny_w_plane`), we want to remove points that belongs to the planar surface, with
- `rng = np.random.default_rng(120)` as random number generator
- distance tolerance = 2e-4 (points less than this distance tolerance should be removed)

**Question:** For the condition above, how many points that belongs to the planar surface should be removed?

In [66]:
rng = np.random.default_rng(120)  # random number generator
bunny_wo_plane2 = remove_plane(bunny_w_plane, ransac, rng, tol=2e-4)
bunny_w_plane.shape[1] - bunny_wo_plane2.shape[1]

[0.62340903 0.61632702 0.60924501 ... 0.07225798 0.05039054 0.03572844]
[0.16422836 0.1654593  0.16669025 ... 0.0341678  0.02934603 0.04787035]
[0.68791923 0.66957684 0.65123445 ... 0.07487202 0.04336961 0.04183034]
[0.22856055 0.21092869 0.19329682 ... 0.0474456  0.02690593 0.04592335]
[0.65349544 0.64163662 0.62977779 ... 0.12388994 0.09592295 0.09169807]
[0.42698791 0.4082264  0.38946489 ... 0.0405933  0.01804866 0.02547785]
[0.61093318 0.59201103 0.57308888 ... 0.02276711 0.0092372  0.00816207]
[0.12210207 0.11690623 0.1117104  ... 0.03788035 0.03734212 0.04424731]
[0.20506495 0.18740018 0.16973541 ... 0.02194096 0.00135764 0.02029956]
[0.54190273 0.52166911 0.50143549 ... 0.08137851 0.05113127 0.06077214]
[0.10612572 0.10363368 0.10114165 ... 0.0864406  0.0745236  0.08467961]
[0.1937652  0.19997828 0.20619135 ... 0.06062201 0.05691419 0.0452136 ]
[0.21048735 0.20453016 0.19857297 ... 0.05030863 0.05310275 0.06367946]
[0.47917254 0.46013404 0.44109554 ... 0.04054816 0.01643126 0.02

2512